## 不带先验约束项的图像分割

In [ ]:
from skimage.segmentation import chan_vese
from skimage.filters import gaussian
from skimage import io, img_as_float
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2
import math

def load_image(path):
    if path.endswith('.mat'):
        mat = sio.loadmat(path)
        # assume the image is the first non-meta variable
        for k in mat:
            if not k.startswith('__'):
                img = mat[k]
                break
        img = img.astype(np.float64)
    else:
        img = img_as_float(io.imread(path, as_gray=True))
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

def init_LSF(img, SEED=None):
    IniLSF = np.ones((img.shape[0],img.shape[1]),img.dtype) 
    if SEED is None:
        centre = (img.shape[0]//2,img.shape[1]//2)
    elif isinstance(SEED, int):
        np.random.seed(SEED)
        random_x = np.random.randint(img.shape[0]//4, img.shape[0]-img.shape[0]//4)
        random_y = np.random.randint(img.shape[1]//4, img.shape[1]-img.shape[1]//4)
        centre = (random_x, random_y)
    elif isinstance(SEED, tuple):
        centre = SEED
    else:
        raise ValueError("SEED must be an integer or a tuple of (x, y) coordinates")
    IniLSF[centre[0]-img.shape[0]//4:centre[0]+img.shape[0]//4,
           centre[1]-img.shape[1]//4:centre[1]+img.shape[1]//4] = -1
    IniLSF=-IniLSF
    return IniLSF

def cv_segmentation(img, IniLSF='chekerboard'):
    img_smooth = gaussian(img, sigma=1)
    cv_result = chan_vese(img_smooth, mu=0.08, lambda1=1, lambda2=1, tol=1e-6, 
                          dt=0.5, init_level_set=IniLSF, extended_output=True)
    
    segmented, phi, energies = cv_result

    return segmented, phi, energies

def cv_segmentation_multiphase_adaptive(img, IniLSF='chekerboard', intensity_threshold=0.1):    
    # 预处理：高斯平滑
    img_smooth = gaussian(img, sigma=1)
    
    # 初始分割
    cv_result = chan_vese(
        img_smooth,
        mu=0.07,
        lambda1=0.6,
        lambda2=1,
        tol=1e-5,
        dt=0.5,
        init_level_set=IniLSF,
        extended_output=True
    )
    
    segmented, phi, energies = cv_result
    
    # 分析分割区域的强度分布
    foreground_intensity = img[segmented > 0]
    
    # 检查是否需要进行多相分割
    fg_std = np.std(foreground_intensity) if len(foreground_intensity) > 0 else 0
    
    segmented_multi = np.zeros_like(img, dtype=np.int32)
    phis = [phi]
    energies_list = [energies]
    
    current_phase = 1
    fg_seg = segmented
    
    while fg_std > intensity_threshold and len(foreground_intensity) > 10 and current_phase < 5:
        mask = fg_seg > 0
        # 取亮的为前景
        fg_mask = mask if np.mean(img[mask]) > np.mean(img[~mask]) else ~mask
        fg_img = img.copy()
        # 前景轮廓处的均值代替背景值
        foreground_contour = np.logical_and(fg_mask, np.logical_xor(phi > 0, np.roll(phi, 1, axis=0) > 0))
        fg_img[~fg_mask] = np.mean(img[foreground_contour])
        # 重新归一化
        fg_img = (fg_img - fg_img.min()) / (fg_img.max() - fg_img.min() + 1e-8)

        cv_fg = chan_vese(
            fg_img,
            mu=0.02,
            lambda1=0.8,
            lambda2=1,
            tol=2e-6,
            dt=0.3,
            init_level_set=IniLSF,
            extended_output=True
        )
        
        fg_seg, fg_phi, fg_energies = cv_fg
        mask = fg_seg > 0
        # 取亮的为前景
        new_mask = mask if np.mean(img[mask]) > np.mean(img[~mask]) else ~mask
        segmented_multi[fg_mask & ~new_mask] = current_phase
        current_phase += 1
        segmented_multi[fg_mask & new_mask] = current_phase
        current_phase += 1
        
        phis.append(fg_phi)
        energies_list.append(fg_energies)

        foreground_intensity = img[new_mask]
        fg_std = np.std(foreground_intensity) if len(foreground_intensity) > 0 else 0
        phi = fg_phi
    if current_phase == 1:
        # 如果不需要多相分割，返回二值结果
        segmented_multi = segmented.astype(np.int32)
    
    return segmented_multi, phis, energies_list

def mat_math(img, intput, type):
    output=intput 
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            if type=="atan":
                output[i,j] = math.atan(intput[i,j]) 
            if type=="sqrt":
                output[i,j] = math.sqrt(intput[i,j]) 
    return output 

def _rsf_iteration(LSF, img, mu=0.08, nu=0.003 * 255 * 255, epison=1, 
                    step=0.1, lambda1=1, lambda2=0.9, sig=16):
    kernel = np.ones((sig*4+1,sig*4+1),np.float64)/(sig*4+1)**2 
    Drc = (epison / math.pi) / (epison*epison+ LSF*LSF) 
    Hea = 0.5*(1 + (2 / math.pi)*mat_math(img,LSF/epison,"atan")) 
    Iy, Ix = np.gradient(LSF) 
    s = mat_math(img,Ix*Ix+Iy*Iy,"sqrt") 
    Nx = Ix / (s+0.000001) 
    Ny = Iy / (s+0.000001) 
    Mxx,Nxx =np.gradient(Nx) 
    Nyy,Myy =np.gradient(Ny) 
    cur = Nxx + Nyy 
    Length = nu*Drc*cur 

    Lap = cv2.Laplacian(LSF,-1) 
    Penalty = mu*(Lap - cur) 

    KIH = cv2.filter2D(Hea*img,-1,kernel) 
    KH = cv2.filter2D(Hea,-1,kernel) 
    f1 = KIH / KH  
    KIH1 = cv2.filter2D((1-Hea)*img,-1,kernel) 
    KH1 = cv2.filter2D(1-Hea,-1,kernel) 
    f2 = KIH1 / KH1  
    R1 = (lambda1- lambda2)*img*img 
    R2 = cv2.filter2D(lambda1*f1 - lambda2*f2,-1,kernel) 
    R3 = cv2.filter2D(lambda1*f1*f1 - lambda2*f2*f2,-1,kernel) 
    RSFterm = -Drc*(R1-2*R2*img+R3) 

    LSF = LSF + step*(Length + Penalty + RSFterm) 
    #plt.imshow(s, cmap ='gray'),plt.show() 
    return LSF

def rsf_segmentation(img, IniLSF, n=300):
    img = gaussian(img, sigma=1)
    LSF = IniLSF
    img = np.uint8(img * 255)
    for i in range(1,n):
        LSF = _rsf_iteration(LSF, img)
    segmented = LSF > 0
    return segmented.astype(np.int32), LSF, None

def plot_results(img, iniLSF, segmented, phis, filename = None):
    plt.rcParams['font.sans-serif']=['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    
    plt.figure(figsize=(13, 5))
    plt.rcParams['font.size'] = 20
    colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k']

    # 初始轮廓线
    plt.subplot(1, 3, 1)
    plt.imshow(img, cmap='gray')
    if isinstance(iniLSF, list):
        for i, phi in enumerate(iniLSF):
            plt.contour(phi, [0], colors=colors[i])
    else:
        plt.contour(iniLSF, [0], colors='r')
    plt.title('初始轮廓线')
    plt.axis('off')

    # 分割结果
    plt.subplot(1, 3, 2)
    plt.imshow(segmented, cmap='gray')
    plt.title('分割结果')
    plt.axis('off')

    # 轮廓曲线与原始图像
    plt.subplot(1, 3, 3)
    plt.imshow(img, cmap='gray')
    if isinstance(phis, list):
        for i, phi in enumerate(phis):
            plt.contour(phi, [0], colors=colors[i])
    else:
        plt.contour(phis, [0], colors='r')
    plt.title('最终轮廓线')
    plt.axis('off')

    # plt.tight_layout()
    if filename is not None:
        if not os.path.exists(os.path.dirname(filename)):
            os.makedirs(os.path.dirname(filename))
        plt.savefig(filename)
    else:
        plt.savefig('segmented.png')

    plt.show()

In [ ]:
import numpy as np
import scipy.ndimage as ndi
from scipy import signal
import matplotlib.pyplot as plt

def heaviside(x, epsilon=1.0):
    """光滑的Heaviside函数"""
    return 0.5 * (1 + (2 / np.pi) * np.arctan(x / epsilon))

def dirac_delta(x, epsilon=1.0):
    """光滑的Dirac delta函数"""
    return epsilon / (np.pi * (epsilon**2 + x**2))

def gaussian_kernel(sigma, size=None):
    """创建高斯核"""
    if size is None:
        size = int(2 * np.ceil(3 * sigma) + 1)
    kernel_1d = np.linspace(-(size // 2), size // 2, size)
    kernel_1d = np.exp(-0.5 * (kernel_1d / sigma) ** 2)
    kernel_2d = np.outer(kernel_1d, kernel_1d)
    return kernel_2d / np.sum(kernel_2d)

def edge_detection_function(grad_u0, beta=100):
    """边缘检测函数g"""
    return 1.0 / (1.0 + beta * grad_u0 ** 2)

def compute_gradient_magnitude(img):
    """计算图像梯度幅值"""
    grad_y, grad_x = np.gradient(img)
    return np.sqrt(grad_x**2 + grad_y**2)

def compute_local_contrast(img, window_size=3):
    """计算局部对比度 LCR_W"""
    from scipy.ndimage import maximum_filter, minimum_filter
    
    V_max = maximum_filter(img, size=window_size)
    V_min = minimum_filter(img, size=window_size)
    V_g = 1.0  # 对于归一化图像，最大强度为1.0
    
    LCR_W = (V_max - V_min) / V_g
    return LCR_W

def vector_shrinkage(x, y, g=None):
    """向量值shrinkage算子"""
    if g is not None:
        y = g / y
    
    norm = np.maximum(np.sqrt(x[0]**2 + x[1]**2), 1e-8)
    shrink = np.maximum(1.0 - y / norm, 0.0)
    
    result = np.zeros_like(x)
    result[0] = shrink * x[0]
    result[1] = shrink * x[1]
    
    return result

def gauss_seidel_step(s, d, b, lamda, phi_shape, bound_min=-2, bound_max=2):
    """Gauss-Seidel迭代更新水平集函数"""
    phi = np.zeros(phi_shape)
    d_x = d[0]
    d_y = d[1]
    b_x = b[0]
    b_y = b[1]
    
    # 扩展数组边界以便处理边界像素
    phi_ext = np.zeros((phi_shape[0]+2, phi_shape[1]+2))
    d_x_ext = np.pad(d_x, 1, mode='edge')
    d_y_ext = np.pad(d_y, 1, mode='edge')
    b_x_ext = np.pad(b_x, 1, mode='edge')
    b_y_ext = np.pad(b_y, 1, mode='edge')
    s_ext = np.pad(s, 1, mode='edge')
    
    for i in range(1, phi_shape[0]+1):
        for j in range(1, phi_shape[1]+1):
            # 计算 alpha
            alpha = (d_x_ext[i-1, j] - d_x_ext[i, j] + 
                     d_y_ext[i, j-1] - d_y_ext[i, j] -
                     (b_x_ext[i-1, j] - b_x_ext[i, j] + 
                      b_y_ext[i, j-1] - b_y_ext[i, j]))
            
            # 计算 beta
            beta = 0.25 * (phi_ext[i-1, j] + phi_ext[i+1, j] + 
                           phi_ext[i, j-1] + phi_ext[i, j+1] - 
                           (1.0/lamda) * s_ext[i, j] + alpha)
            
            # 限制在[-2, 2]范围内
            phi_ext[i, j] = np.clip(beta, bound_min, bound_max)
    
    # 提取内部区域
    phi = phi_ext[1:-1, 1:-1]
    
    return phi

def automatic_local_global_active_contour(img, IniLSF, max_iter=50, tol=1e-3, 
                                         sigma=18, lamda1=15.4, lamda2=14, 
                                         lamda_split=0.04, gamma=1e-2, 
                                         beta=100, window_size=3, filter_sigma=1):
    """
    自动结合局部与全局信息的活动轮廓模型
    
    参数:
        img: 归一化的输入图像 (0-1范围)
        iniLSF: 初始化水平集函数 (在初始轮廓内为2，外部为-2)
        max_iter: 最大迭代次数
        tol: 收敛容差
        sigma: 高斯核标准差
        lamda1, lamda2: 拟合能量权重参数
        lamda_split: Split Bregman参数
        gamma: 权函数参数
        beta: 边缘检测函数参数
        window_size: 局部对比度窗口大小
    
    返回:
        segmented: 二值分割结果
        LSF: 最终水平集函数
    """
    # 高斯滤波
    img = ndi.gaussian_filter(img, sigma=filter_sigma)
    
    # 初始化变量
    phi = IniLSF.copy()
    u0 = img.copy()
    
    # 图像尺寸
    rows, cols = img.shape
    
    # 初始化辅助变量
    d = np.zeros((2, rows, cols))  # d = (d_x, d_y)
    b = np.zeros((2, rows, cols))  # b = (b_x, b_y)
    
    # 创建高斯核
    kernel_size = int(2 * np.ceil(3 * sigma) + 1)
    K_sigma = gaussian_kernel(sigma, kernel_size)
    
    # 计算图像梯度幅值
    grad_mag = compute_gradient_magnitude(u0)
    
    # 计算边缘检测函数g
    g = edge_detection_function(grad_mag, beta)
    
    # 计算局部对比度
    LCR_W = compute_local_contrast(u0, window_size)
    avg_LCR_W = np.mean(LCR_W)
    
    # 迭代计数器
    iter_count = 0
    phi_old = phi.copy()
    
    phi_change_history = 0.0

    while iter_count < max_iter:
        # 1. 计算M1和M2
        M1 = heaviside(phi)
        M2 = 1.0 - M1
        
        # 2. 更新拟合函数f1, f2和均值常量c1, c2
        # f1 = (Kσ * (M1 * u0)) / (Kσ * M1)
        numerator_f1 = signal.convolve2d(M1 * u0, K_sigma, mode='same', boundary='symm')
        denominator_f1 = signal.convolve2d(M1, K_sigma, mode='same', boundary='symm')
        f1 = numerator_f1 / (denominator_f1 + 1e-8)
        
        # f2 = (Kσ * (M2 * u0)) / (Kσ * M2)
        numerator_f2 = signal.convolve2d(M2 * u0, K_sigma, mode='same', boundary='symm')
        denominator_f2 = signal.convolve2d(M2, K_sigma, mode='same', boundary='symm')
        f2 = numerator_f2 / (denominator_f2 + 1e-8)
        
        # c1 = ∫(u0 * M1) / ∫M1
        c1 = np.sum(u0 * M1) / (np.sum(M1) + 1e-8)
        
        # c2 = ∫(u0 * M2) / ∫M2
        c2 = np.sum(u0 * M2) / (np.sum(M2) + 1e-8)
        
        # 3. 计算权函数ω (根据式3-18)
        # ω = γ * average(LCR_W) * (1 - LCR_W)
        omega = gamma * avg_LCR_W * (1 - LCR_W)
        
        # 4. 计算局部强度拟合力F1和全局强度拟合力F2 (式3-7)
        # 预先计算一些项
        # 对于每个位置x，我们需要计算卷积项
        # 由于这涉及到双重积分，我们使用卷积来高效计算
        
        # 计算局部项
        term1_local = np.zeros_like(u0)
        term2_local = np.zeros_like(u0)
        
        for i in range(rows):
            for j in range(cols):
                # 局部窗口（简化实现）
                x = u0[i, j]
                
                # 局部拟合项
                # ∫ Kσ(y-x) |u0(x) - f1(y)|^2 dy
                local_window = max(1, kernel_size // 2)
                i_min = max(0, i - local_window)
                i_max = min(rows, i + local_window + 1)
                j_min = max(0, j - local_window)
                j_max = min(cols, j + local_window + 1)
                
                # 提取局部区域
                y_coords = np.arange(i_min, i_max)
                x_coords = np.arange(j_min, j_max)
                YY, XX = np.meshgrid(y_coords, x_coords, indexing='ij')
                
                # 计算核权重
                distances = np.sqrt((YY - i)**2 + (XX - j)**2)
                kernel_weights = np.exp(-0.5 * (distances / sigma)**2)
                kernel_weights = kernel_weights / np.sum(kernel_weights)
                
                # 计算局部积分
                term1_val = np.sum(kernel_weights * (x - f1[YY, XX])**2)
                term2_val = np.sum(kernel_weights * (x - f2[YY, XX])**2)
                
                term1_local[i, j] = term1_val
                term2_local[i, j] = term2_val
        
        # 计算全局项
        term1_global = (u0 - c1)**2
        term2_global = (u0 - c2)**2
        
        # 计算F1和F2
        F1 = (1 - omega) * (-lamda1 * term1_local + lamda2 * term2_local)
        F2 = omega * (-lamda1 * term1_global + lamda2 * term2_global)
        
        # 5. 计算s(x) = -(F1 + F2)
        s = -(F1 + F2)
        
        # 6. Split Bregman迭代
        # 6.1 更新水平集函数φ
        phi = gauss_seidel_step(s, d, b, lamda_split, phi.shape)
        
        # 6.2 更新辅助变量d
        # 计算∇φ
        phi_grad_y, phi_grad_x = np.gradient(phi)
        phi_grad = np.stack([phi_grad_x, phi_grad_y])
        
        # shrink_g(b^k + ∇φ^{k+1}, 1/λ)
        d = vector_shrinkage(b + phi_grad, 1.0/lamda_split, g)
        
        # 6.3 更新Bregman变量b
        b = b + phi_grad - d
        
        # 7. 检查收敛条件
        phi_change = np.linalg.norm(phi - phi_old) / np.sqrt(rows * cols)
        
        if phi_change < tol or abs(phi_change_history - phi_change) < 1e-2*tol:
            print(f"收敛于迭代 {iter_count}, 变化: {phi_change}")
            break
        
        phi_old = phi.copy()
        phi_change_history = phi_change
        iter_count += 1
        
        if iter_count % 10 == 0:
            print(f"迭代 {iter_count}, 变化: {phi_change}")
    
    print(f"完成 {iter_count} 次迭代")
    
    # 根据水平集函数生成分割结果
    if img[phi > 0].mean() > img[phi <= 0].mean():
        segmented = np.zeros_like(img, dtype=np.uint8)
        segmented[phi > 0] = 1
    else:
        segmented = np.zeros_like(img, dtype=np.uint8)
        segmented[phi <= 0] = 1

    return segmented, phi, None

In [ ]:
# 测试图像
test_images = [
    'special_image.bmp',
    '4.bmp',
    'synthetic_test.png',
    '2.bmp', 
    '3.bmp',
    '1.bmp',
    '5.bmp',
    '6.png',
    'brain_img75.mat',
    'myBrain_axial.bmp',
]

# 模型
models = {
    'CV': cv_segmentation,
    'multi_CV': cv_segmentation_multiphase_adaptive,
    'RSF': rsf_segmentation,
    'LGAC': automatic_local_global_active_contour,
}

def test(model_name, init='chekerboard'):
    model_func = models[model_name]
    for img_file in test_images:
        print(f'Running {model_name} on {img_file}...')
        img = load_image('CV&RSF实验图\\'+img_file)
        if init=='chekerboard':
            _, IniLSF, _ = chan_vese(img, max_num_iter=0, extended_output=True)
        else:
            IniLSF = init_LSF(img, init)
        segmented, phi, energies = model_func(img, IniLSF=IniLSF)
        init_type = init if init is not None else 'centre'
        plot_results(img, IniLSF, segmented, phi, filename = f'results\\{model_name}\\{init_type}\\'+img_file.split('.')[0]+'_segmented.png')

def test_all():
    for init in ['chekerboard', None]:
        for model_name, model_func in models.items():
            print(f'Running {model_name}...')
            for img_file in test_images:
                img = load_image('CV&RSF实验图\\'+img_file)
                if init=='chekerboard':
                    _, IniLSF, _ = chan_vese(img, max_num_iter=0, extended_output=True)
                else:
                    IniLSF = init_LSF(img, init)
                segmented, phi, energies = model_func(img, IniLSF=IniLSF)
                init_type = init if init is not None else 'centre'
                plot_results(img, IniLSF, segmented, phi, filename = f'results\\{model_name}\\{init_type}\\'+img_file.split('.')[0]+'_segmented.png')

In [ ]:
test_images = [
    'ISIC_0011345.jpg', 
    'leftventricle5.bmp',
    'melanoma1.jpg',
    '19021.jpg',
    '208001.jpg'
]
test('RSF', None)

In [ ]:
test_image = '1.bmp'
label_image = '1-l.bmp'
# 网格搜索参数调优
from sklearn.model_selection import ParameterGrid

# 水平集函数的距离
def distance(phi, label_LSF):
    return np.sum((phi - label_LSF)**2)

label_img = load_image('CV&RSF实验图\\'+label_image)
label_LSF = (1 - label_img > 1e-6).astype(np.float32)
img = load_image('CV&RSF实验图\\'+test_image)
plt.imshow(img, cmap='gray')
plt.imshow(label_LSF, cmap='gray')
plt.contour(label_LSF, levels=[0.5], colors='r')
plt.axis('off')
plt.show()
# LGAC模型参数
param_grid = {
    'lamda_split': [2.8e-2, 3e-2, 3.2e-2],
}
IniLSF = init_LSF(img, None)
results = []
for param in ParameterGrid(param_grid):
    # param['lamda2'] = param['lamda1'] * 10/11
    print(f'Testing parameters: {param}')
    segmented, phi, energies = automatic_local_global_active_contour(img, IniLSF, **param)
    plot_results(img, IniLSF, segmented, phi)
    dist = distance(segmented, label_LSF)
    print(f'Distance: {dist}')
    results.append((param, dist))

best_param, best_dist = min(results, key=lambda x: x[1])
from pprint import pprint
pprint(results)
print(f'Best parameters: {best_param}, Distance: {best_dist}')

## 带先验约束项的图像分割

In [ ]:
import numpy as np
import scipy.ndimage as ndi
from scipy import signal
import matplotlib.pyplot as plt

def heaviside(x, epsilon=1.0):
    """光滑的Heaviside函数"""
    return 0.5 * (1 + (2 / np.pi) * np.arctan(x / epsilon))

def dirac_delta(x, epsilon=1.0):
    """光滑的Dirac delta函数"""
    return epsilon / (np.pi * (epsilon**2 + x**2))

def gaussian_kernel(sigma, size=None):
    """创建高斯核"""
    if size is None:
        size = int(2 * np.ceil(3 * sigma) + 1)
    kernel_1d = np.linspace(-(size // 2), size // 2, size)
    kernel_1d = np.exp(-0.5 * (kernel_1d / sigma) ** 2)
    kernel_2d = np.outer(kernel_1d, kernel_1d)
    return kernel_2d / np.sum(kernel_2d)

def edge_detection_function(grad_u0, beta=100):
    """边缘检测函数g"""
    return 1.0 / (1.0 + beta * grad_u0 ** 2)

def compute_gradient_magnitude(img):
    """计算图像梯度幅值"""
    grad_y, grad_x = np.gradient(img)
    return np.sqrt(grad_x**2 + grad_y**2)

def vector_shrinkage(x, y, g=None):
    """向量值shrinkage算子（式4-7）"""
    if g is not None:
        y = g / y
    
    norm = np.maximum(np.sqrt(x[0]**2 + x[1]**2), 1e-8)
    shrink = np.maximum(1.0 - y / norm, 0.0)
    
    result = np.zeros_like(x)
    result[0] = shrink * x[0]
    result[1] = shrink * x[1]
    
    return result

def compute_rsf_data_term(img, phi, sigma=3.0, lambda1=1e-5, lambda2=1e-5):
    """
    计算RSF模型的数据项T(x) = p1(x) + p2(x)（式4-3）
    
    参数:
        img: 输入图像
        phi: 当前水平集函数
        sigma: 高斯核标准差
        lambda1, lambda2: 权重参数
    
    返回:
        T: 数据项
        f1, f2: 局部强度近似函数
    """
    
    # 计算Heaviside函数和掩模
    H_phi = heaviside(phi)
    M1 = H_phi
    M2 = 1.0 - H_phi
    
    # 创建高斯核
    kernel_size = int(2 * np.ceil(3 * sigma) + 1)
    K_sigma = gaussian_kernel(sigma, kernel_size)
    
    # 计算局部强度近似函数f1和f2（RSF模型）
    # f1 = (Kσ * (M1 * I)) / (Kσ * M1)
    numerator_f1 = signal.convolve2d(M1 * img, K_sigma, mode='same', boundary='symm')
    denominator_f1 = signal.convolve2d(M1, K_sigma, mode='same', boundary='symm')
    f1 = numerator_f1 / (denominator_f1 + 1e-8)
    
    # f2 = (Kσ * (M2 * I)) / (Kσ * M2)
    numerator_f2 = signal.convolve2d(M2 * img, K_sigma, mode='same', boundary='symm')
    denominator_f2 = signal.convolve2d(M2, K_sigma, mode='same', boundary='symm')
    f2 = numerator_f2 / (denominator_f2 + 1e-8)
    
    # 计算p1(x)和p2(x)（式4-3）
    # p1(x) = λ1 ∫ Kσ(y-x) |I(x) - f1(y)|^2 dy
    # p2(x) = -λ2 ∫ Kσ(y-x) |I(x) - f2(y)|^2 dy
    
    rows, cols = img.shape
    p1 = np.zeros_like(img)
    p2 = np.zeros_like(img)
    
    # 为了高效计算，我们可以将双重积分简化为卷积运算
    # 注意：这是一个近似，精确计算需要双重积分
    # 在实际应用中，这种近似通常足够好
    
    # 计算局部窗口大小
    local_window = max(1, kernel_size // 2)
    
    for i in range(rows):
        for j in range(cols):
            # 提取局部窗口
            i_min = max(0, i - local_window)
            i_max = min(rows, i + local_window + 1)
            j_min = max(0, j - local_window)
            j_max = min(cols, j + local_window + 1)
            
            # 创建局部坐标网格
            y_coords = np.arange(i_min, i_max)
            x_coords = np.arange(j_min, j_max)
            YY, XX = np.meshgrid(y_coords, x_coords, indexing='ij')
            
            # 计算高斯核权重
            distances = np.sqrt((YY - i)**2 + (XX - j)**2)
            kernel_weights = np.exp(-0.5 * (distances / sigma)**2)
            kernel_weights = kernel_weights / np.sum(kernel_weights)
            
            # 当前像素强度
            I_x = img[i, j]
            
            # 计算p1(x)
            f1_vals = f1[YY, XX]
            p1_val = np.sum(kernel_weights * (I_x - f1_vals)**2)
            p1[i, j] = lambda1 * p1_val
            
            # 计算p2(x)
            f2_vals = f2[YY, XX]
            p2_val = np.sum(kernel_weights * (I_x - f2_vals)**2)
            p2[i, j] = -lambda2 * p2_val
    
    # 计算数据项T(x)
    T = p1 + p2
    
    return T, f1, f2

def prior_constraint_active_contour(img, preLSF, iniLSF, max_iter=20, tol=1e-2,
                                   sigma=20, lambda1=16.5, lambda2=15,
                                   lambda_split=0.05, alpha=10, beta=100, sigma_filter=3.4):
    """
    结合先验约束项的图像分割模型
    
    参数:
        img: 归一化的输入图像 (0-1范围)
        preLSF: 预分割的水平集函数（先验约束项Φ_pre）
        iniLSF: 初始化水平集函数
        max_iter: 最大迭代次数
        tol: 收敛容差
        sigma: 高斯核标准差
        lambda1, lambda2: RSF数据项权重
        lambda_split: Split Bregman参数λ
        alpha: 先验约束项权重
        beta: 边缘检测函数参数
    
    返回:
        segmented: 二值分割结果
        LSF: 最终水平集函数
    """
    # 初始化变量
    phi = iniLSF.copy()
    phi_pre = preLSF.copy()
    u0 = img.copy()

    # 高斯滤波
    u0 = ndi.gaussian_filter(u0, sigma=sigma_filter)
    
    # 图像尺寸
    rows, cols = img.shape
    
    # 初始化辅助变量
    s = np.zeros((2, rows, cols))  # s = (s_x, s_y)
    h = np.zeros((2, rows, cols))  # h = (h_x, h_y)
    
    # 计算图像梯度幅值
    grad_mag = compute_gradient_magnitude(u0)
    
    # 计算边缘检测函数g
    g = edge_detection_function(grad_mag, beta)
    
    # 迭代计数器
    iter_count = 0
    phi_change_history = 0
    phi_old = phi.copy()
    
    # Split Bregman迭代
    while iter_count < max_iter:
        # 1. 计算数据项T(x)（式4-3）
        T, f1, f2 = compute_rsf_data_term(u0, phi, sigma, lambda1, lambda2)
        
        # 2. Split Bregman迭代
        # 2.1 固定s，求解Φ的子问题（式4-6）
        # 根据PDF第7页的离散化公式
        phi_new = np.zeros_like(phi)
        
        for i in range(1, rows-1):
            for j in range(1, cols-1):
                # 计算v1: s_x(i-1,j) - s_x(i,j) + s_y(i,j-1) - s_y(i,j)
                v1 = (s[0, i-1, j] - s[0, i, j] + 
                      s[1, i, j-1] - s[1, i, j])
                
                # 计算v2: h_x(i-1,j) - h_x(i,j) + h_y(i,j-1) - h_y(i,j)
                v2 = (h[0, i-1, j] - h[0, i, j] + 
                      h[1, i, j-1] - h[1, i, j])
                
                # 计算v3: φ(i-1,j) + φ(i+1,j) + φ(i,j-1) + φ(i,j+1)
                v3 = (phi[i-1, j] + phi[i+1, j] + 
                      phi[i, j-1] + phi[i, j+1])
                
                # 计算ξ = v1 - v2
                xi = v1 - v2
                
                # 计算β = (λ * v3 - T(i,j) + λ * ξ) + α * Φ_pre(i,j)
                beta_val = (lambda_split * v3 - T[i, j] + 
                           lambda_split * xi + alpha * phi_pre[i, j])
                
                # 计算φ_new(i,j) = β / (α + 4λ)
                phi_new[i, j] = beta_val / (alpha + 4 * lambda_split)
        
        # 处理边界（使用边界复制）
        phi_new[0, :] = phi_new[1, :]
        phi_new[-1, :] = phi_new[-2, :]
        phi_new[:, 0] = phi_new[:, 1]
        phi_new[:, -1] = phi_new[:, -2]
        
        # 2.2 固定Φ，求解s的子问题（式4-7）
        # 计算∇φ_new
        phi_grad_y, phi_grad_x = np.gradient(phi_new)
        phi_grad = np.stack([phi_grad_x, phi_grad_y])
        
        # shrink_g(h^k + ∇φ^{k+1}, 1/λ) = shrink(h^k + ∇φ^{k+1}, g/λ)
        s = vector_shrinkage(h + phi_grad, 1.0/lambda_split, g)
        
        # 2.3 更新Bregman变量h（式4-6最后一行）
        h = h + phi_grad - s
        
        # 3. 更新水平集函数
        phi = phi_new
        
        # 4. 检查收敛条件
        phi_change = np.linalg.norm(phi - phi_old) / np.sqrt(rows * cols)
        
        if phi_change < tol or abs(phi_change_history - phi_change) < 1e-2*tol:
            print(f"收敛于迭代 {iter_count}, 变化: {phi_change}")
            break
        
        phi_old = phi.copy()
        phi_change_history = phi_change
        iter_count += 1
        
        if iter_count % 10 == 0:
            print(f"迭代 {iter_count}, 变化: {phi_change}")
    
    print(f"完成 {iter_count} 次迭代")
    
    # 根据水平集函数生成分割结果
    segmented = np.zeros_like(img, dtype=np.uint8)
    segmented[phi > 0] = 1
    
    # segmented = eliminate_small_regions(segmented, 100)
    
    return segmented, phi

import numpy as np
from scipy import ndimage
from typing import Tuple, Optional

def eliminate_small_regions(matrix: np.ndarray, 
                           area_threshold: int, 
                           connectivity: int = 4) -> np.ndarray:
    """
    消除矩阵中面积不大于阈值的连续区域
    
    参数:
        matrix: 浮点数矩阵
        area_threshold: 面积阈值，小于等于此值的区域将被消除
        connectivity: 连通性定义，4或8连通（默认4）
        
    返回:
        处理后的矩阵
    """
    
    # 创建二值掩码：0值区域为0，非0值区域为1
    binary_mask = (matrix > 0).astype(np.uint8)
    
    # 获取连通组件
    if connectivity == 4:
        structure = ndimage.generate_binary_structure(2, 1)
    elif connectivity == 8:
        structure = ndimage.generate_binary_structure(2, 2)
    else:
        raise ValueError("connectivity必须是4或8")
    
    # 标记0区域（背景）
    zero_mask = (matrix <= 0).astype(np.uint8)
    labeled_zero, num_zero_regions = ndimage.label(zero_mask, structure=structure)
    
    # 标记1区域（前景）
    one_mask = (matrix > 0).astype(np.uint8)
    labeled_one, num_one_regions = ndimage.label(one_mask, structure=structure)
    
    # 创建结果矩阵的副本
    result = matrix.copy()
    
    # 处理0区域：小面积0区域变为1
    zero_sizes = ndimage.sum(zero_mask, labeled_zero, range(num_zero_regions + 1))
    for label in range(1, num_zero_regions + 1):
        if zero_sizes[label] <= area_threshold:
            # 找到该区域的位置并设置为1（使用矩阵均值或1.0）
            region_mask = (labeled_zero == label)
            # 使用周围非0值的均值，如果没有则使用1.0
            if np.any(~region_mask):
                # 获取区域边界值
                result[region_mask] = np.mean(matrix[~region_mask & (matrix > 0)]) if np.any(matrix[~region_mask] > 0) else 1.0
            else:
                result[region_mask] = 1.0
    
    # 处理1区域：小面积1区域变为0
    one_sizes = ndimage.sum(one_mask, labeled_one, range(num_one_regions + 1))
    for label in range(1, num_one_regions + 1):
        if one_sizes[label] <= area_threshold:
            # 找到该区域的位置并设置为0
            region_mask = (labeled_one == label)
            result[region_mask] = 0.0
    
    return result

In [ ]:
def plot_results(img, iniLSF, preLSF, segmented, phis, filename = None):
    plt.rcParams['font.sans-serif']=['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    
    plt.figure(figsize=(13, 5))
    plt.rcParams['font.size'] = 20

    # 初始轮廓线
    plt.subplot(1, 3, 1)
    plt.imshow(img, cmap='gray')
    plt.contour(iniLSF, [0], colors='r')
    plt.contour(preLSF, [0], colors='b')
    plt.title('初始轮廓线与先验约束项')
    plt.axis('off')

    # 分割结果
    plt.subplot(1, 3, 2)
    plt.imshow(segmented, cmap='gray')
    plt.title('分割结果')
    plt.axis('off')

    # 轮廓曲线与原始图像
    plt.subplot(1, 3, 3)
    plt.imshow(img, cmap='gray')
    plt.contour(phis, [0], colors='r')
    plt.title('最终轮廓线')
    plt.axis('off')

    # plt.tight_layout()
    if filename is not None:
        if not os.path.exists(os.path.dirname(filename)):
            os.makedirs(os.path.dirname(filename))
        plt.savefig(filename)
    else:
        plt.savefig('segmented.png')

    plt.show()

In [ ]:
dataset = '基于先验约束信息的图像分割图像'
types = ['.png', '.jpg', '.bmp']
images = {}
for file in os.listdir(dataset):
    if '-l' in file:
        name = file.split('-l')[0]
        for type in types:
            if os.path.exists(dataset+'/'+name+type):
                images[name+type] = file
print(images)

In [ ]:
from sklearn.model_selection import ParameterGrid
param = ParameterGrid({
    'sigma': [5],
    'lambda1': [5.5,],
    'lambda_split': [0.006,],
    'alpha': [0.4],
    'beta': [10],
    'sigma_filter': [1.6],
})
for image_file, label_file in images.items():
    img = load_image('基于先验约束信息的图像分割图像\\'+image_file)
    preLSF = load_image('基于先验约束信息的图像分割图像\\'+label_file)
    preLSF = (1 - preLSF < 1e-6).astype(np.float32)

    # 初始化水平集函数
    iniLSF = init_LSF(img, None)

    # 应用结合先验约束项的活动轮廓模型
    print(f"正在处理 {image_file}...")
    for params in param:
        if 'lambda1' in params:
            params['lambda2'] = params['lambda1'] * 10/11
            params['tol'] = params['lambda1'] * 5e-4
        print(params)
        segmented, final_lsf = prior_constraint_active_contour(
            img, preLSF, iniLSF, **params
        )
        # 绘制结果
        plot_results(img, iniLSF, preLSF, segmented, final_lsf, filename='results\\TCM\\'+image_file.split('.')[0]+'_segmented.png')
        a_lsf = eliminate_small_regions(final_lsf, 3e-2*min(np.count_nonzero(segmented), np.count_nonzero(1 - segmented)))
        segmented = a_lsf > 0
        plot_results(img, iniLSF, preLSF, segmented, a_lsf, filename='results\\TCM_a\\'+image_file.split('.')[0]+'_segmented.png')

### 从水平集到智能分割：图像分割实验的实践、演进与思考

完成《图像分割》课程系列实验，如同亲历了一段浓缩的技术演进史。从最初对Chan-Vese（CV）模型公式的推演，到最终实现结合先验约束的复杂模型，这不仅是代码与算法实现的过程，更是一场对“如何让机器理解图像内容”这一核心问题的深度探索。通过亲手复现、调试并对比从CV到RSF，再到自动结合局部与全局信息模型，以及最终引入先验约束的模型，我对图像分割领域的经典脉络与内在逻辑有了血肉丰满的认识，也对技术发展的驱动力和未来方向产生了自己的感想与展望。

**启程：CV模型的简洁之美与现实的初次碰撞**

一切始于CV模型。其能量泛函的优雅与简洁令人印象深刻——仅用两个全局常数来代表前景与背景，辅以曲线长度约束，便能驱动一条闭合轮廓自动演化并分割出目标。在实验初期，面对强度均匀的合成图像，CV模型的表现堪称完美，轮廓光滑，收敛迅速，让我直观感受到了基于区域的活动轮廓模型相对于传统边缘检测方法的优势：它不苛求清晰的梯度，而着眼于区域属性的“同质性”。

然而，这种美感很快在复杂现实图像面前显露出局限。当处理那张经典的、带有明显亮度渐变的合成图像（synthetic_test.bmp）时，CV模型彻底“失明”了。无论我如何调整参数或更换初始轮廓（从中心矩形到复杂的棋盘格），它都无法准确提取左下角的暗色圆形区域。究其根源，在于其“分段常数”的核心假设在强度不均匀的现实中过于理想化。这第一次“挫败”并非徒劳，它深刻地教育了我：一个模型的有效性紧密依赖于其前提假设与待解决问题的匹配程度。CV模型是一把精良的尺子，但它只能测量平整的桌面，对于起伏的地形则无能为力。这次实践让我明白，脱离实际场景空谈模型优劣是没有意义的，识别问题的本质（如强度不均匀）是选择乃至改进模型的第一步。

**进阶：RSF模型的局部洞察与新的平衡难题**

于是，RSF模型登场，带来了“局部拟合”这一强大思想。通过引入高斯核函数，模型不再用僵化的全局均值，而是为图像中每一点动态计算其局部邻域内的前景/背景强度估计。实现过程中，计算局部拟合函数 `f1(x)`, `f2(x)` 的卷积操作，让我第一次感受到计算复杂度的显著上升，但也亲眼见证了性能的飞跃。那张让CV模型折戟的合成图像，在RSF模型下被清晰、准确地分割出来。在血管图像上，RSF能够捕捉到那些微弱、断续的边缘，这是CV模型完全无法做到的。

然而，RSF也非万能。实验中最深刻的体会莫过于其对初始轮廓的敏感性和在复杂背景前的犹豫。例如，在部分大脑影像中，使用简单的中心矩形初始化，轮廓有时会被背景的复杂纹理“吸住”，陷入错误的局部极小值。而改用棋盘格初始化虽常能改善，却又可能在另一些图像上导致对背景的过分割。这揭示了RSF模型的一个核心矛盾：高度依赖局部信息使其能适应不均匀性，但也使之更容易受到局部噪声或初始位置的误导，缺乏全局的稳健性。此外，手动调节高斯核尺度σ等参数，更像一门“艺术”，需要针对不同图像反复尝试。这个过程让我认识到，提升模型某一方面的能力（如处理不均匀性）往往会引入新的挑战（如对初始化和参数的敏感），而一个优秀模型需要在多个目标（精度、鲁棒性、自动化）之间取得艰难而精巧的平衡。

**融合与自动化：探索更智能的范式**

“自动结合局部与全局信息的活动轮廓模型”实验，则是一次向更高阶智能的迈进。这个模型的设计理念令我激赏：它不再要求使用者预先决定是相信局部还是全局信息，而是通过一个自适应的权函数 `ω(x)`，让模型根据图像自身的局部对比度动态决策。在平坦区域，权函数增大，模型行为趋近于稳健的CV模型；在边缘附近，权函数减小，模型则展现出RSF般的局部灵敏性。

实现此模型的过程最具挑战，也最具启发性。除了要实现RSF的局部拟合和CV的全局计算，还需编码自适应权函数和基于Split Bregman的凸优化求解器。调参过程尤为曲折，尤其是`λ_split`和`γ`等参数，其最优值在不同图像间差异巨大。例如，为血管图像找到的最佳参数组合，直接套用到大脑图像上效果可能一落千丈。这让我切身体会到，尽管模型在机制上追求“自适应”，但在超参数层面仍难以完全摆脱“人工调优”的窠臼。不过，当看到模型在血管图像上，以比RSF更稳定的方式，同时保持了高精度的分割效果时，我感受到了算法进步的实在价值。它验证了一个重要思想：将不同模型的优势通过一种数据驱动的方式有机融合，是提升综合性能的有效路径。

**引入先验：当模型开始“学习”与“合作”**

第四个实验——“结合先验约束项的图像分割模型”，为我打开了另一扇窗：如何将外部知识（先验）系统地注入分割过程。在心脏左心室或肝脏肿瘤的分割任务中，提供一个粗略的预分割（如来自阈值法或甚至医生简单的标注）作为约束，模型的分割效果发生了质变。原本可能扩散到整个器官的轮廓，被有效地“锚定”在目标附近。这让我强烈感受到，在许多专业领域（尤其是医学），完全“无中生有”的通用分割是极其困难的，而引入领域知识（先验）能极大地缩小解空间，提升结果的可靠性和准确性。

然而，实现中也暴露了新问题：先验的质量至关重要。一个粗糙或不准确的预分割，反而会“误导”模型。同时，先验约束项权重 `α` 的设定极为关键，过强会扼杀轮廓应有的演化动力，导致轮廓无法脱离先验走向真实更精确的边界；过弱则又失去约束意义。这引申出一个更深层的思考：如何自动化地获取高质量的先验？这自然地将我的目光引向了当下如火如荼的深度学习。

**展望：深度融合时代的切身感受与未来遐想**

回顾整个实验历程，从CV到带先验的模型，我切身感受到图像分割技术演进的清晰脉络：**从全局到局部，再从局部到局部与全局的自适应结合；从纯粹依赖底层图像特征，到主动融入高层先验知识；从独立的数学模型，到与学习框架的初步对接。** 每一次演进，都是为了解决前一代模型在更复杂、更真实场景中暴露的核心痛点。

而深度学习，特别是像U-Net这样的编码器-解码器架构，代表着另一种范式革命。在我的了解与对比中，深度学习模型通过海量数据训练，能够隐式地学习从图像到分割图的端到端复杂映射，其优势在于对复杂特征和上下文信息的强大抽取能力，且在拥有充足标注数据时，能实现令人惊叹的自动化分割性能。这与我们实验的传统水平集方法形成鲜明对比：后者基于严谨的数学模型和可解释的能量最小化过程，更具透明性和可控性，尤其擅长处理小样本、弱边界和需要融入明确物理/几何约束（如平滑性、形状先验）的场景。

在我看来，二者并非取代关系，而是走向融合。未来的趋势，很可能如我们最后一个实验所预示的那样，是**“深度学习的感知能力”与“水平集方法的优化框架与可解释性”的强强联合**。例如，用U-Net快速生成高质量的先验轮廓或概率图，作为水平集模型的初始化或约束项；或者，将深度网络学习到的特征嵌入到水平集能量函数的数据项中。当前一些前沿研究，如将水平集演化作为深度学习网络中的可微分层，实现端到端的训练，正是这一方向的积极探索。

**个人感想与收获**

通过这一系列的动手实践，我最大的收获不仅是弄懂了几个模型的原理和代码，更是建立了一种“模型思维”。我学会了如何像一个研究者一样去思考：面对一个分割任务，先分析其核心挑战（强度不均？弱边界？噪声多？少样本？）；然后评估不同模型的假设与这些挑战的匹配度；接着在实践中验证，并敏锐观察失败案例，从而理解模型的真实边界何在。调试参数时的挫败，改进算法后的欣喜，对比结果时的了然，这些都是书本和论文无法给予的宝贵经验。

图像分割模型的演进史，是一部不断打破假设、拥抱复杂性的历史。从假设强度均匀，到承认局部变化；从假设无需先验，到积极利用知识。作为学习者，我深刻体会到，在人工智能领域，没有一个模型是终极的。技术的进步正是在解决旧问题、发现新问题、再寻求新方案的循环中螺旋上升。展望未来，我期待看到更多融合了深度学习强大表示能力和传统模型可解释性、严谨性的新型混合架构出现。而这段从CV公式推导到思考AI融合的实践旅程，无疑为我今后无论是继续学术研究，还是投身工业应用，都奠定了坚实而富有洞察力的基础。在数据与算法驱动的时代，理解工具背后的“为什么”与“如何演进”，或许比单纯会使用工具更为重要。